In [ ]:
# Load packages
from pathlib import Path
import os
import re
import sys
import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from skyline_qc import *


In [ ]:

# Skyline data path
skyline_path = "data/MARTHA/DE17501_Martha_results_dotp.csv"

# Import skyline data (includes column "Isotope Label Type" from Precursor)
skyline_importer = ImportFile(skyline_path)
skyline_data = skyline_importer.import_skyline_file()

In [ ]:
# Isotope Label Type is added in ImportFile.import_skyline_file() (from Precursor).

# qREPs spike levels
qREPs_spike_levels = pd.read_csv('ratio/DE17501_ratio.csv')

# SDRF
sdrf_path = 'sdrf/sdrf_MARTHA_combined.sdrf.tsv'
sdrf_data = pd.read_csv(sdrf_path, sep='\t')



In [ ]:
# cross_check_skyline_sdrf(skyline_df = skyline_data, sdrf_df = sdrf_data)

In [ ]:
# Import and use function from output_test.py to get iRT peptides
iRT_peptides = get_irt_peptides(skyline_data)

print('This is the iRT peptides:')
print(iRT_peptides)

In [ ]:
qc_samples = skyline_importer.suggest_qc_samples(skyline_data)

skyline_checker = CheckSkylineFile(skyline_path)
qc_data = skyline_checker.get_qc_data(qc_samples, skyline_data)
# Suggest qc samples
print('This is the qc samples:')
print(f'Number of qc samples: {len(qc_samples)}')
print(qc_samples)

In [ ]:
# Get test samples (exclude QC samples from all replicate names)
skyline_checker = CheckSkylineFile(skyline_path)
test_samples = skyline_checker.get_test_samples(qc_samples, skyline_data)
test_data = skyline_checker.get_test_data(test_samples, skyline_data)
print('This is the test samples:')
print(f'Number of test samples: {len(test_samples)}')

In [ ]:


# Show all Replicate in df here
qc_replicates = qc_samples
print(qc_replicates)
print(f'Number of QC replicates: {len(qc_replicates)}')


# Remove all rows in skyline_data that contains any of the qc_replicates
skyline_data = test_data
print(test_data.shape)

In [ ]:
plot_library_dot_product_distribution(skyline_data)

In [ ]:
skyline_clean = filter_library_dot_product(skyline_data, threshold=0.6)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = summarise_peptide_counts(skyline_clean)


In [ ]:
report_summary, peptide_list = report_peptide_protein_summary(peptide_counts)

In [ ]:
plot_heavy_light_scatter(peptide_counts)

In [ ]:
filtered_peptide_counts = filter_peptide_counts(peptide_counts, light_cutoff=700, heavy_cutoff=700)

filtered_peptide_counts.head()


In [ ]:
selected_peptides_report,selected_peptides = report_peptide_protein_summary(filtered_peptide_counts)

In [ ]:
from skyline_qc.importer import MergeFiles

skyline_merge = MergeFiles(skyline_data, sdrf_data, selected_peptides).merge_files()


In [ ]:
skyline_merge.head()

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 

pool_data = skyline_merge[skyline_merge['characteristics[Sample]'] == 'Pool']
# First, sort dataframe by Plate
pool_data = pool_data.sort_values('characteristics[Plate]')
# Reset index
pool_data = pool_data.reset_index(drop=True)
pool_data.head()

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
plot_pool_boxplot(pool_data)


In [ ]:
# Calculate intra-plate CV

peptide_plate_stats = calculate_intra_plate_cv(pool_data, col_name='characteristics[Plate]')
plot_intra_plate_cv_stats(peptide_plate_stats, col_name='characteristics[Plate]')

In [ ]:
# Example usage:
plot_inter_plate_cv_kde(peptide_plate_stats)

In [ ]:
# Example usage:
interplate_cv = calculate_inter_plate_cv(peptide_plate_stats)
interplate_cv.head()

In [ ]:
plot_cumulative_peptide_count_by_cv(interplate_cv)

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 
# using only peptides in the top 10% lowest inter-plate CV for normalization

def extract_top_percentile(df, column, percentile=0.1, id_col='Peptide Sequence', source_df=None, source_col=None):
    """
    Extract unique IDs from `id_col` where values in `column` are at or below the given percentile,
    and return both the ID list and filtered DataFrame from `source_df` (if provided).

    Args:
        df (pd.DataFrame): DataFrame containing summary/statistics (e.g. interplate_cv).
        column (str): Name of column to compute percentile threshold over (e.g. 'inter_plate_cv').
        percentile (float): Fraction for percentile threshold (e.g. 0.1 for 10% lowest values).
        id_col (str): Column in `df` whose unique values to extract (peptide identifier).
        source_df (pd.DataFrame, optional): DataFrame to filter based on the returned ID list.
        source_col (str, optional): Column of `source_df` to match IDs (default: id_col).

    Returns:
        tuple: (ID list, filtered DataFrame [if source_df given, else None])
    """
    threshold = df[column].quantile(percentile)
    id_list = df[df[column] <= threshold][id_col].unique()
    if source_df is not None:
        if source_col is None:
            source_col = id_col
        filtered_df = source_df[source_df[source_col].isin(id_list)]
        return id_list, filtered_df.reset_index(drop=True)
    else:
        return id_list, None

top10cv_peptides, pool_selected_df = extract_top_percentile(
    interplate_cv,
    column='inter_plate_cv',
    percentile=0.1,
    id_col='Peptide Sequence',
    source_df=pool_data,
    source_col='Peptide Sequence'
)

# Filter pool_data to include only those peptides for normalization calculation
pool_selected_df

In [ ]:
# Example usage:
model, anova_res = plate_peptide_anova(pool_selected_df)

In [ ]:
anova_res


In [ ]:
def fit_plate_logratio_model(selected_norm_peptides):
    """
    Fits a linear model log(RatioLightToHeavy) ~ Plate (Plate as categorical).
    Returns the fitted model and the cleaned DataFrame (with Plate and log_ratio columns).

    Args:
        selected_norm_peptides (pd.DataFrame): DataFrame containing at least 'characteristics[Plate]', 'Replicate', and 'RatioLightToHeavy'
    
    Returns:
        model: statsmodels OLS fitted model
        df: DataFrame with columns ['Plate', 'Replicate', 'RatioLightToHeavy', 'log_ratio', ...]
    """
    # Ensure required columns exist
    required_cols = ['characteristics[Plate]', 'RatioLightToHeavy', 'Replicate']
    for col in required_cols:
        if col not in selected_norm_peptides.columns:
            raise ValueError(f"Missing required column: '{col}'")

    df = selected_norm_peptides.rename(
        columns={'characteristics[Plate]': 'Plate'}
    ).copy()

    # Ensure numeric RatioLightToHeavy
    df['RatioLightToHeavy'] = pd.to_numeric(df['RatioLightToHeavy'], errors='coerce')
    df = df.dropna(subset=['RatioLightToHeavy', 'Plate'])

    # Create log_ratio column
    df['log_ratio'] = np.log(df['RatioLightToHeavy'])

    # Model: log(ratio) ~ Plate (as categorical)
    m = smf.ols('log_ratio ~ C(Plate)', data=df).fit()
    print(m.summary())

    return m, df


def plot_logratio_by_plate_boxplot(df):
    """
    Plots boxplots of log(RatioLightToHeavy) by Replicate, colored by Plate.

    Args:
        df (pd.DataFrame): DataFrame with columns ['Replicate', 'log_ratio', 'Plate']
    """
    # Sort dataframe by Plate for plotting (optional)
    df_sorted = df.sort_values('Plate')
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Replicate', y='log_ratio', data=df_sorted, hue='Plate', dodge=False)
    plt.title('Boxplot of log(RatioLightToHeavy) by Replicate (colored by Plate)')
    plt.xlabel('')
    plt.ylabel('log(RatioLightToHeavy)')
    plt.xticks([], [])  # Remove x tick labels and marks
    plt.legend(title='Plate', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Example usage:
model, df = fit_plate_logratio_model(selected_norm_peptides)
plot_logratio_by_plate_boxplot(df)

In [ ]:
# Example usage:
conv_factors_df, conversion_factors, m = get_plate_conversion_factors(df)

In [ ]:
def adjust_ratio_by_plate(df, conversion_factors):
    """
    Normalize RatioLightToHeavy by dividing by the plate-specific conversion factor.

    Args:
        df (pd.DataFrame): DataFrame with columns 'RatioLightToHeavy' and 'Plate'.
        conversion_factors (dict): Dict mapping plate (as str or int) to factor.

    Returns:
        pd.DataFrame: Input DataFrame with added 'RatioLightToHeavy_adj' column.
    """
    def get_factor(plate):
        plate_str = str(int(plate)) if str(int(plate)) in conversion_factors else str(plate)
        return conversion_factors[plate_str]

    df = df.copy()
    df['RatioLightToHeavy_adj'] = [
        r / get_factor(p) for r, p in zip(df['RatioLightToHeavy'], df['Plate'])
    ]
    return df

# Example usage:
adjusted_df = adjust_ratio_by_plate(df, conversion_factors)
adjusted_df


In [ ]:
# Use the adjust_ratio_by_plate function above to add RatioLightToHeavy_adj to skyline_merge
# First, construct a Plate column of the correct type for the function
skyline_merge_adj = skyline_merge.copy()
# Make sure 'Plate' column exists and matches conversion_factors keys
skyline_merge_adj['Plate'] = skyline_merge_adj['characteristics[Plate]']
skyline_merge_adj = adjust_ratio_by_plate(skyline_merge_adj, conversion_factors)


In [ ]:
# From skyline_merge_adj, filter to pool samples
pool_data_adj = skyline_merge_adj[skyline_merge_adj['characteristics[Sample]'] == 'Pool'].copy()

# If your peptide column is 'Peptide Sequence', rename it once for convenience
if 'Peptide Sequence' in pool_data_adj.columns and 'Peptide' not in pool_data_adj.columns:
    pool_data_adj = pool_data_adj.rename(columns={'Peptide Sequence': 'Peptide'})

# Filter to only peptides in selected_peptides
pool_data_adj = pool_data_adj[pool_data_adj['Peptide'].isin(selected_peptides)].copy() 



In [ ]:

# Plot boxplot of log-transformed RatioLightToHeavy_adj for each Replicate, colored by Plate,
# with hidden x-axis labels and in the style of the previous plot format
df_sorted_adj = pool_data_adj.sort_values('characteristics[Plate]')
plt.figure(figsize=(12, 8))
sns.boxplot(
    x='Replicate',
    y='RatioLightToHeavy_adj',
    data=df_sorted_adj,
    hue='characteristics[Plate]'
)
plt.title('Boxplot of log(RatioLightToHeavy_adj) for Pool samples by Replicate (colored by Plate)')
plt.xlabel('')
plt.ylabel('log(RatioLightToHeavy_adj)')
plt.yscale('log')
plt.xticks([], [])  # Remove x tick labels and marks
plt.legend(title='Plate', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


# Calculate absolute quantification

In [ ]:

# Add new column call qRePS which is the string before _ in Protein Name col
skyline_merge_adj['qRePS'] = skyline_merge_adj['Protein Name'].str.split('_').str[0]

In [ ]:
# Merge qREPs_spike_levels, to skyline_merge_adj by qRePs
skyline_merge_adj = pd.merge(skyline_merge_adj, qREPs_spike_levels, on=['qRePS'], how='left')
# Add new column call qRePs which is the string before _ in Protein Name col

# Add new column called Protein conc [pmol] which is equal to RatioLightToHeavy_adj * Amount Per well [pmol]
skyline_merge_adj['Protein conc [pmol]'] = skyline_merge_adj['RatioLightToHeavy_adj'] * skyline_merge_adj['Amount per well [pmol]']

# Round Protein conc [pmol] to 4 decimal places
skyline_merge_adj['Protein conc [pmol]'] = skyline_merge_adj['Protein conc [pmol]'].round(4)


In [ ]:
# Export wide format for qREPs


# Select columns to export
export_cols = ['Replicate', 'Peptide Sequence', 'Protein Name', 'qRePS', 'Protein conc [pmol]']

# Pivot the dataframe to wide format
skyline_merge_adj_wide = skyline_merge_adj.pivot(
    index=['qRePS', 'Peptide Sequence', 'Protein Name'],
    columns='Replicate',
    values='Protein conc [pmol]'
)

# Export to csv status
# skyline_merge_adj_wide.to_csv('export/MARTHA_conc_normalized.csv', index=True)


